# A3.3 · Filesystem and path guards

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

---

**Risk.** The agent iterating on code wanders into credentials.

**Control.** Workspace scoping, mount discipline, ephemeral state.

**This lab.** Stop the agent wandering from code into credentials.

| | |
|---|---|
| Open-source tooling | Docker, Kyverno |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A3.3"))

Path guards fail in one specific way, over and over: the check runs before normalisation, and `workspace/../../secret` starts with `workspace/`.

In [ ]:
from cybercommons import sandbox

g = sandbox.PathGuard(workspace="/work")
paths = ["/work/src/main.py",
         "/work/./src/../src/main.py",
         "/work/../../root/.ssh/id_rsa",
         "/work/.env",
         "/work/deploy.pem",
         "/etc/shadow",
         "/work/sub/../.aws/credentials"]
for p in paths:
    print(g.check(p))

Now the buggy version, to see the failure rather than read about it.

In [ ]:
def naive_check(path, workspace="/work"):
    return path.startswith(workspace)          # the bug

for p in paths:
    real = g.check(p).allowed
    naive = naive_check(p)
    flag = "  ← NAIVE CHECK IS WRONG" if naive != real else ""
    print(f"{p:38s} guard={str(real):5s} naive={str(naive):5s}{flag}")

### Expect

The guard allows the two legitimate workspace paths and denies the rest, naming the resolved path or the deny rule. The naive check wrongly allows the traversal to `/root/.ssh/id_rsa` and the `.aws/credentials` read.

### Your turn

Symlinks are the next layer: a link inside the workspace pointing out of it defeats pure string normalisation. What has to change — and which real containment technology gives it to you for free?

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A3.3.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*